In [ ]:
# Trains and resumes YOLO26m and YOLO11m models for child detection.

In [ ]:
!pip install ultralytics

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import yaml
import shutil


# 1. Retrouver automatiquement best (1).pt
matches = (
    list(Path("/kaggle/input").rglob("last (1).pt"))
    + list(Path("/kaggle/working").rglob("last (1).pt"))
)

print("Checkpoints trouvés :")

for path in matches:
    print(path)

assert len(matches) >= 1, (
    "Le fichier best (1).pt n’a pas été trouvé."
)

uploaded_checkpoint = matches[0]


# 2. Copier le checkpoint avec un nom simple
resume_checkpoint = Path(
    "/kaggle/working/yolo26m_resume.pt"
)

shutil.copy2(
    uploaded_checkpoint,
    resume_checkpoint,
)

print(
    "Checkpoint utilisé :",
    resume_checkpoint,
)


# 3. Recréer data.yaml
original_yaml = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/data.yaml"
)

assert original_yaml.exists(), (
    f"Dataset ChildDetection introuvable : {original_yaml}"
)

with open(
    original_yaml,
    "r"
) as file:
    data = yaml.safe_load(file)

data["train"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/train/images"
)

data["val"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/valid/images"
)

data["test"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/test/images"
)

with open(
    "/kaggle/working/data.yaml",
    "w"
) as file:
    yaml.safe_dump(
        data,
        file
    )


# 4. Reprendre l’entraînement
model_26m = YOLO(
    str(resume_checkpoint)
)

results = model_26m.train(
    resume=True
)

In [ ]:
# ============================================================
# SAUVEGARDER BEST.PT ET LAST.PT DANS UN DATASET KAGGLE
# À EXÉCUTER APRÈS AVOIR ARRÊTÉ L'ENTRAÎNEMENT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import shutil
import subprocess
import sys


# ------------------------------------------------------------
# 1. Installer/importer KaggleHub
# ------------------------------------------------------------

try:
    import kagglehub
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "kagglehub",
        ]
    )

    import kagglehub


# ------------------------------------------------------------
# 2. Rechercher le couple best.pt / last.pt le plus récent
# ------------------------------------------------------------

RUNS_ROOT = Path(
    "/kaggle/working/runs"
)

assert RUNS_ROOT.exists(), (
    f"Dossier introuvable : {RUNS_ROOT}"
)

valid_weights_directories = []

for best_path in RUNS_ROOT.rglob(
    "best.pt"
):
    weights_directory = best_path.parent
    last_path = weights_directory / "last.pt"

    if last_path.exists():
        valid_weights_directories.append(
            weights_directory
        )


print("Dossiers de checkpoints trouvés :")

for directory in valid_weights_directories:
    print(directory)


assert len(valid_weights_directories) > 0, (
    "Aucun dossier contenant à la fois "
    "best.pt et last.pt n’a été trouvé."
)


def checkpoint_date(weights_directory):

    best_path = (
        weights_directory
        / "best.pt"
    )

    last_path = (
        weights_directory
        / "last.pt"
    )

    return max(
        best_path.stat().st_mtime,
        last_path.stat().st_mtime,
    )


WEIGHTS_DIRECTORY = max(
    valid_weights_directories,
    key=checkpoint_date,
)

BEST_PATH = (
    WEIGHTS_DIRECTORY
    / "best.pt"
)

LAST_PATH = (
    WEIGHTS_DIRECTORY
    / "last.pt"
)


# ------------------------------------------------------------
# 3. Vérifier que les fichiers ne sont pas vides
# ------------------------------------------------------------

for checkpoint_path in [
    BEST_PATH,
    LAST_PATH,
]:

    assert checkpoint_path.exists(), (
        f"Fichier absent : {checkpoint_path}"
    )

    assert checkpoint_path.stat().st_size > 1_000_000, (
        f"Le checkpoint semble incomplet : "
        f"{checkpoint_path}"
    )


print("\nCheckpoints sélectionnés :")

print(
    "BEST :",
    BEST_PATH,
)

print(
    "LAST :",
    LAST_PATH,
)

print(
    "Taille best.pt :",
    f"{BEST_PATH.stat().st_size / 1024**2:.2f} Mo",
)

print(
    "Taille last.pt :",
    f"{LAST_PATH.stat().st_size / 1024**2:.2f} Mo",
)


# ------------------------------------------------------------
# 4. Préparer le dossier à envoyer
# ------------------------------------------------------------

UPLOAD_DIRECTORY = Path(
    "/kaggle/working/"
    "yolo26m_child_detection_checkpoints2"
)

if UPLOAD_DIRECTORY.exists():
    shutil.rmtree(
        UPLOAD_DIRECTORY
    )

UPLOAD_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    BEST_PATH,
    UPLOAD_DIRECTORY
    / "best.pt",
)

shutil.copy2(
    LAST_PATH,
    UPLOAD_DIRECTORY
    / "last.pt",
)


# ------------------------------------------------------------
# 5. Ajouter un fichier d’informations
# ------------------------------------------------------------

upload_time = datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S UTC"
)

information_text = (
    "YOLO26m Child Detection Checkpoints\n"
    "===================================\n\n"
    f"Date de sauvegarde : {upload_time}\n"
    f"Dossier source : {WEIGHTS_DIRECTORY}\n"
    f"Best source : {BEST_PATH}\n"
    f"Last source : {LAST_PATH}\n\n"
    "Utilisation :\n"
    "- best.pt : inférence et intégration dans le Module 3\n"
    "- last.pt : reprise exacte de l'entraînement\n"
)

(
    UPLOAD_DIRECTORY
    / "checkpoint_information.txt"
).write_text(
    information_text,
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Vérifier le contenu avant envoi
# ------------------------------------------------------------

print("\nFichiers préparés pour le dataset :")

for path in sorted(
    UPLOAD_DIRECTORY.iterdir()
):
    print(
        path.name,
        "-",
        f"{path.stat().st_size / 1024**2:.2f} Mo",
    )


# ------------------------------------------------------------
# 7. Créer ou mettre à jour le dataset Kaggle
# ------------------------------------------------------------

DATASET_HANDLE = (
    "nisrineamalkhnifass/"
    "yolo26m-child-detection-checkpoints"
)

VERSION_NOTES = (
    "Sauvegarde de best.pt et last.pt "
    f"le {upload_time}"
)


print(
    "\nEnvoi vers le dataset :",
    DATASET_HANDLE,
)

kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIRECTORY),
    version_notes=VERSION_NOTES,
)


# ------------------------------------------------------------
# 8. Confirmation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CHECKPOINTS SAUVEGARDÉS SUR KAGGLE")
print("=" * 80)

print(
    "Dataset :",
    DATASET_HANDLE,
)

print(
    "Fichiers envoyés :"
)

print("- best.pt")
print("- last.pt")
print("- checkpoint_information.txt")

In [ ]:
from ultralytics import YOLO
import yaml


# ------------------------------------------------------------
# Préparer le fichier data.yaml
# ------------------------------------------------------------

with open(
    "/kaggle/input/datasets/nisrineamalkhnifass/childdetection/data.yaml",
    "r"
) as f:
    data = yaml.safe_load(f)


data["train"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/train/images"
)

data["val"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/valid/images"
)

data["test"] = (
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "childdetection/full_dataset/full_dataset/test/images"
)


with open(
    "/kaggle/working/data.yaml",
    "w"
) as f:
    yaml.dump(data, f)


# ------------------------------------------------------------
# Entraîner uniquement YOLO26m
# ------------------------------------------------------------

model_26m = YOLO(
    "yolo26m.pt"
)


results = model_26m.train(
    data="/kaggle/working/data.yaml",

    epochs=200,
    imgsz=640,
    batch=8,

    optimizer="Adam",
    lr0=1e-4,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,

    fliplr=0.5,
    hsv_v=0.4,
    hsv_s=0.7,
    hsv_h=0.015,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,

    # Seulement best.pt et last.pt
    save=True,
    save_period=-1,

    # Ne pas créer de cache sur le disque
    cache=False,

    project="/kaggle/working/runs/detect",
    name="yolo26m_200ep",
    exist_ok=True,

    device=0,
    workers=4,
    verbose=True,
)

In [ ]:
from ultralytics import YOLO
import yaml


with open('/kaggle/input/datasets/nisrineamalkhnifass/childdetection/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/train/images'
data['val']   = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/valid/images'
data['test']  = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/test/images'

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)

model_11m = YOLO('yolo11m.pt')
model_11m.train(
    data='/kaggle/working/data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    optimizer='Adam',
    lr0=1e-4,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    fliplr=0.5,
    hsv_v=0.4,
    hsv_s=0.7,
    hsv_h=0.015,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    name='yolo11m_200ep'
)



In [ ]:
from ultralytics import YOLO
import yaml

# Préparer data.yaml
with open('/kaggle/input/datasets/nisrineamalkhnifass/childdetection/data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/train/images'
data['val']   = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/valid/images'
data['test']  = '/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/test/images'

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)
   
model_26m = YOLO('yolo26m.pt')

model_26m.train(
    data='/kaggle/working/data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    optimizer='Adam',
    lr0=1e-4,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    fliplr=0.5,
    hsv_v=0.4,
    hsv_s=0.7,
    hsv_h=0.015,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    name='yolo26m_200ep'
)